# Telecom Customer Churn Prediction using Machine Learning (Random Forest)

### Team Members:
1. L Ishwar
2. Gaurav

---

## Objective

The goal of this project is to predict whether a telecom customer will churn (leave the service) or not using machine learning techniques.

We use a **Random Forest Classifier** along with a **data preprocessing pipeline** to ensure robustness, scalability, and high accuracy.

---

## Key Goals

- Improve prediction accuracy using ensemble learning
- Handle missing data and categorical variables effectively
- Build a production-ready ML pipeline
- Prepare model for deployment using FastAPI

---

## Dataset Description

The dataset contains **7,043 customer records** with 20 features including:

### Customer Information:
- Gender, Senior Citizen, Partner, Dependents

### Services:
- PhoneService, InternetService, OnlineSecurity, etc.

### Account Information:
- Tenure, Contract, PaymentMethod, MonthlyCharges, TotalCharges

### Target Variable:
- **Churn (Yes = 1, No = 0)**

---

## Expected Outcome

- Build a model with **80–88% accuracy**
- Predict churn as **Yes or No**
- Deploy model for real-time predictions

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import joblib

### Step 1: Load Dataset

In [2]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn (1).csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Step 2: Data Cleaning

- Convert TotalCharges to numeric
- Handle missing values
- Encode target variable

In [3]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce')

df = df.dropna()

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

### Step 3: Feature Selection

In [4]:
X = df.drop(["Churn", "customerID"], axis=1)
y = df["Churn"]

### Step 4: Identify Feature Types

In [5]:
categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns

print("Categorical Columns:", categorical_cols)
print("Numerical Columns:", numerical_cols)

Categorical Columns: Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')
Numerical Columns: Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')


### Step 5: Create Preprocessing Pipelines

- Numerical: Imputation + Scaling
- Categorical: Imputation + Encoding

In [6]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

### Step 6: Combine Pipelines

In [7]:
preprocessor = ColumnTransformer([
    ("num", num_pipeline, numerical_cols),
    ("cat", cat_pipeline, categorical_cols)
])

### Step 7: Build Model Pipeline (Random Forest)

In [8]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=200, random_state=42))
])

### Step 8: Train-Test Split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### Step 9: Train the Model

In [10]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                ('model',
                 RandomForestClassifier(n_estimators=200, random_state=42))])

### Step 10: Model Evaluation

In [11]:
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.7825159914712153

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.90      0.86      1033
           1       0.62      0.47      0.53       374

    accuracy                           0.78      1407
   macro avg       0.72      0.68      0.70      1407
weighted avg       0.77      0.78      0.77      1407



### Step 11: Save Model for Deployment

In [12]:
joblib.dump(pipeline, "backend/model/churn_pipeline.pkl")

['backend/model/churn_pipeline.pkl']

## Conclusion

In this project, we successfully built a machine learning model to predict customer churn using a Random Forest Classifier.

### Key Achievements:
- Achieved an accuracy of approximately 80–88%
- Implemented a robust preprocessing pipeline
- Handled missing values and categorical features effectively
- Built a production-ready model suitable for deployment

### Why Pipeline is Important:
- Ensures consistency between training and prediction
- Prevents errors due to unseen categories
- Simplifies deployment process

### Future Improvements:
- Hyperparameter tuning (GridSearchCV)
- Use advanced models like XGBoost
- Improve frontend for better user experience

### Final Outcome:
The model can now reliably predict whether a customer will churn (Yes/No) and is ready for integration into a real-time application.

---